# Học hồi quy với dữ liệu giá nhà California

Notebook này dạy từng bước: xử lý giá trị thiếu và cột chữ bằng pipeline, linear regression, Ridge, Lasso, KNN regression và Gradient Boosting.

Dữ liệu: 20.640 khu dân cư (block group) ở California theo điều tra dân số năm 1990. Mỗi dòng là một khu, không phải một căn nhà. Biến mục tiêu là median_house_value, giá trị nhà trung vị của khu (USD). Nguồn: bản CSV trong sách Hands-On Machine Learning của Aurélien Géron, có thêm cột ocean_proximity và một số ô trống so với bản trong scikit-learn.

Chuẩn bị:
1. Đặt file data4.csv cùng thư mục với notebook.
2. Cài thư viện nếu chưa có: pip install pandas numpy matplotlib seaborn scikit-learn

Cách làm: đọc phần giải thích, viết code vào ô trống, chạy bằng Shift + Enter, rồi so sánh với kết quả mong đợi. Nếu kết quả khác, đọc lại đề bài và thử lại trước khi xem đáp án.

## Bước 0: Import thư viện

Mục tiêu: nạp các thư viện dùng trong toàn bộ notebook. Ô này đã viết sẵn, bạn chỉ cần chạy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

## Bước 1: Đọc dữ liệu

Mục tiêu: đưa file CSV vào DataFrame và kiểm tra kích thước.

Kiến thức: pd.read_csv đọc file thành bảng. df.shape trả về (số dòng, số cột).

Kết quả mong đợi: (20640, 10).

In [ ]:
# Viết code ở đây:
# 1. Đọc file data4.csv vào biến df
# 2. In df.shape và 5 dòng đầu


## Bước 2: Kiểu dữ liệu, ô trống và cột chữ

Mục tiêu: tìm cột bị thiếu dữ liệu và xem cột chữ có những giá trị nào.

Kiến thức: isna().sum() đếm ô trống theo từng cột. Cột ocean_proximity là cột chữ (categorical), cần mã hóa trước khi đưa vào mô hình.

Kết quả mong đợi: chỉ total_bedrooms bị thiếu, 207 ô. ocean_proximity có 5 nhóm: <1H OCEAN 9136, INLAND 6551, NEAR OCEAN 2658, NEAR BAY 2290 và ISLAND chỉ 5 khu.

Câu hỏi: một nhóm chỉ có rất ít mẫu (ISLAND) sẽ gây vấn đề gì khi chia train/test?

In [ ]:
# Viết code ở đây:
# 1. Gọi df.info()
# 2. In số ô trống của từng cột (chỉ in cột có ô trống)
# 3. In value_counts của ocean_proximity


## Bước 3: Phân phối biến mục tiêu

Mục tiêu: xem giá nhà phân bố thế nào và phát hiện giá trị bị chặn trên.

Kiến thức: khi thu thập dữ liệu, giá trị trên một ngưỡng có thể bị ghi bằng đúng ngưỡng đó (capping). Ở đây, mọi khu có giá trên 500.000 USD bị ghi là 500.001. Mô hình sẽ không học được giá thật của các khu này.

Kết quả mong đợi: giá trung bình khoảng 206.856 USD, trung vị 179.700 USD, thấp nhất 14.999 USD. Có 965 khu bị chặn ở 500.001 USD, tạo một cột cao bất thường ở cuối histogram.

In [ ]:
# Viết code ở đây:
# 1. In describe() của median_house_value (làm tròn 0 chữ số)
# 2. Đếm số khu có median_house_value >= 500001
# 3. Vẽ histogram với 50 cột


## Bước 4: Bản đồ giá nhà

Mục tiêu: dùng tọa độ để thấy giá nhà phụ thuộc vào vị trí.

Kiến thức: scatter với x là kinh độ và y là vĩ độ cho ra hình bản đồ. Tham số c tô màu theo giá, s đặt kích thước điểm theo dân số.

Kết quả mong đợi: giá cao tập trung ở ven biển, quanh khu vực Los Angeles và vịnh San Francisco. Vùng nội địa phía đông có giá thấp hơn.

In [ ]:
# Viết code ở đây:
# 1. Vẽ plt.scatter: x = longitude, y = latitude, c = median_house_value, cmap='jet', s = population / 100, alpha = 0.4
# 2. Thêm plt.colorbar() và tiêu đề


## Bước 5: Tạo feature mới và xem tương quan

Mục tiêu: tạo các tỷ lệ có ý nghĩa hơn số tổng, rồi xem cột nào liên quan nhất đến giá.

Kiến thức: total_rooms là tổng số phòng của cả khu, nên khu đông dân thì số này lớn mà không nói gì về nhà to hay nhỏ. Chia cho số hộ (households) cho ra số phòng trung bình mỗi hộ. Đây là feature engineering.

Kết quả mong đợi: median_income có tương quan cao nhất, khoảng 0,688. bedrooms_per_room khoảng -0,256, mạnh hơn total_bedrooms (0,050). rooms_per_household khoảng 0,152, mạnh hơn total_rooms (0,134).

Câu hỏi: vì sao bedrooms_per_room có tương quan âm với giá?

In [ ]:
# Viết code ở đây:
# 1. rooms_per_household = total_rooms / households
# 2. bedrooms_per_room = total_bedrooms / total_rooms
# 3. population_per_household = population / households
# 4. Tính tương quan của các cột số với median_house_value, sắp xếp giảm dần, làm tròn 3 chữ số


## Bước 6: Chia train và test

Mục tiêu: giữ lại 20% dữ liệu làm tập test.

Kiến thức: tách y ra trước. Mọi bước xử lý (điền ô trống, chuẩn hóa) sẽ nằm trong pipeline và chỉ học từ tập train.

Kết quả mong đợi: X_train có 16512 dòng và 12 cột, X_test có 4128 dòng.

In [ ]:
# Viết code ở đây:
# 1. y = df['median_house_value'], X = df bỏ cột này
# 2. train_test_split với test_size=0.2, random_state=42
# 3. In kích thước X_train và X_test


## Bước 7: Pipeline tiền xử lý

Mục tiêu: gom việc điền ô trống, chuẩn hóa và one-hot vào một đối tượng duy nhất.

Kiến thức: ColumnTransformer áp dụng các bước khác nhau cho các nhóm cột khác nhau. Cột số: SimpleImputer(strategy="median") điền ô trống bằng trung vị của tập train, rồi StandardScaler. Cột chữ: OneHotEncoder tạo mỗi giá trị thành một cột 0/1. handle_unknown="ignore" giúp không lỗi khi tập test có giá trị chưa gặp.

Kết quả mong đợi: sau khi biến đổi có 16 cột: 11 cột số và 5 cột one-hot của ocean_proximity.

In [ ]:
# Viết code ở đây:
# 1. num_cols = các cột số của X_train (select_dtypes('number')), cat_cols = ['ocean_proximity']
# 2. num_pipe = Pipeline gồm SimpleImputer(strategy='median') và StandardScaler()
# 3. prep = ColumnTransformer([('num', num_pipe, num_cols), ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
# 4. fit_transform trên X_train, in kích thước kết quả và get_feature_names_out()


## Bước 8: Linear regression và baseline

Mục tiêu: có con số đầu tiên để so sánh các mô hình sau.

Kiến thức: RMSE là căn bậc hai của sai số bình phương trung bình, cùng đơn vị với y (USD), phạt nặng các lỗi lớn. MAE là sai số tuyệt đối trung bình. Hàm rmse đã được định nghĩa ở bước 0. Baseline: luôn đoán bằng giá trung bình của tập train.

Kết quả mong đợi: RMSE khoảng 69.127 USD, MAE khoảng 49.645 USD, R² khoảng 0,635. Baseline đoán trung bình có RMSE khoảng 114.486 USD.

In [ ]:
# Viết code ở đây:
# 1. Tạo Pipeline gồm ('prep', prep) và ('model', LinearRegression()), fit trên X_train
# 2. Dự đoán X_test, in RMSE, MAE (làm tròn 0 chữ số) và R² (3 chữ số)
# 3. Tính RMSE của baseline đoán y_train.mean()


## Bước 9: Ridge: phạt hệ số lớn

Mục tiêu: thử regularization L2 với nhiều mức alpha.

Kiến thức: Ridge cộng thêm alpha nhân tổng bình phương các hệ số vào hàm mất mát. alpha = 0 giống linear regression. alpha lớn làm hệ số co về gần 0, mô hình đơn giản hơn. Dùng cross-validation để chọn alpha. scoring="neg_root_mean_squared_error" trả về RMSE mang dấu âm (sklearn luôn coi số lớn hơn là tốt hơn), nên cần đổi dấu.

Kết quả mong đợi: CV RMSE gần như không đổi từ alpha = 0,01 đến alpha = 100 (khoảng 67.850 đến 67.880). Đến alpha = 1000 thì lỗi tăng lên khoảng 69.043.

Câu hỏi: với 16.000 dòng và chỉ 16 cột, bạn có nghĩ regularization giúp nhiều không? Kết quả có khớp với dự đoán của bạn?

In [ ]:
# Viết code ở đây:
# 1. cv = KFold(n_splits=5, shuffle=True, random_state=42)
# 2. Lặp alpha trong [0.01, 0.1, 1, 10, 100, 1000]
# 3. Với mỗi alpha: Pipeline(prep, Ridge(alpha)), tính -cross_val_score(..., scoring='neg_root_mean_squared_error').mean()
# 4. In RMSE theo alpha (làm tròn 0 chữ số)


## Bước 10: Lasso: chọn feature tự động

Mục tiêu: thấy Lasso đưa một số hệ số về đúng 0.

Kiến thức: Lasso dùng phạt L1 (tổng giá trị tuyệt đối của hệ số). Khác với Ridge, L1 có thể đưa hệ số về đúng 0, tức là bỏ hẳn feature đó. Vì vậy Lasso vừa là mô hình vừa là cách chọn feature.

Kết quả mong đợi: alpha = 10 loại 1 hệ số, alpha = 100 loại 2, alpha = 1000 loại 5, alpha = 5000 loại 10 trên 16 hệ số và test RMSE tăng lên khoảng 74.032. Có thể xuất hiện ConvergenceWarning, bạn có thể bỏ qua.

In [ ]:
# Viết code ở đây:
# 1. Lặp alpha trong [10, 100, 1000, 5000]
# 2. Fit Pipeline(prep, Lasso(alpha=alpha, max_iter=10000)) trên X_train
# 3. Lấy hệ số: pipe.named_steps['model'].coef_, đếm số hệ số bằng 0
# 4. In alpha, RMSE trên X_test, số hệ số bằng 0
# 5. Với alpha = 5000, in tên các feature bị loại (dùng prep.get_feature_names_out())


## Bước 11: Đọc hệ số của Ridge

Mục tiêu: biết feature nào làm giá tăng hoặc giảm.

Kiến thức: vì các cột số đã chuẩn hóa, hệ số cho biết giá thay đổi bao nhiêu USD khi feature tăng một độ lệch chuẩn. Có thể so sánh độ lớn các hệ số với nhau. Với cột one-hot, hệ số là chênh lệch so với mức trung bình của các nhóm.

Kết quả mong đợi: median_income có hệ số dương lớn nhất trong các cột số (khoảng 78.904). latitude (khoảng -56.587) và longitude (khoảng -56.222) có hệ số âm lớn nhất. ISLAND có hệ số khoảng 93.365 nhưng chỉ dựa trên vài khu.

Câu hỏi: latitude và longitude đều có hệ số âm lớn. Điều đó có nghĩa gì trên bản đồ ở bước 4?

In [ ]:
# Viết code ở đây:
# 1. Fit Pipeline(prep, Ridge(alpha=1)) trên X_train
# 2. Tạo Series hệ số với index là get_feature_names_out()
# 3. Sắp xếp theo giá trị và in ra (làm tròn 0 chữ số)


## Bước 12: KNN regression

Mục tiêu: dự đoán giá bằng trung bình giá của k khu giống nhất.

Kiến thức: KNeighborsRegressor lấy trung bình y của k điểm gần nhất. Vì dùng khoảng cách, chuẩn hóa là bắt buộc (pipeline prep đã làm việc này). KNN không giả định quan hệ tuyến tính, nên có thể học được hiệu ứng vị trí mà linear regression bỏ sót.

Kết quả mong đợi: k = 10 có CV RMSE thấp nhất, khoảng 60.885. Test RMSE khoảng 60.942, tốt hơn linear regression khoảng 8.000 USD.

In [ ]:
# Viết code ở đây:
# 1. Lặp k trong [3, 5, 10, 20, 50]
# 2. Với mỗi k: tính CV RMSE của Pipeline(prep, KNeighborsRegressor(n_neighbors=k)) như ở bước 9
# 3. In kết quả, chọn best_k có RMSE nhỏ nhất
# 4. Fit với best_k và in test RMSE


## Bước 13: Gradient Boosting

Mục tiêu: huấn luyện mô hình gồm nhiều cây nhỏ học nối tiếp nhau.

Kiến thức: random forest trồng các cây độc lập rồi lấy trung bình. Gradient boosting trồng cây lần lượt: mỗi cây mới học phần sai số (residual) mà các cây trước để lại. learning_rate thu nhỏ đóng góp của mỗi cây, n_estimators là số cây, max_depth thường nhỏ (3 đến 5). Bước này mất khoảng 30 giây đến 1 phút.

Kết quả mong đợi: RMSE khoảng 46.028 USD, MAE khoảng 30.297 USD, R² khoảng 0,838.

In [ ]:
# Viết code ở đây:
# 1. Tạo Pipeline(prep, GradientBoostingRegressor(n_estimators=500, learning_rate=0.1, max_depth=4, random_state=42))
# 2. Fit trên X_train, in test RMSE, MAE và R²


## Bước 14: Số cây và overfitting

Mục tiêu: xem lỗi trên train và test thay đổi thế nào khi thêm cây.

Kiến thức: staged_predict trả về dự đoán sau 1 cây, 2 cây, ..., n cây. Lỗi train luôn giảm. Lỗi test giảm rồi đi ngang hoặc tăng lại khi mô hình bắt đầu overfit. Vì staged_predict thuộc về mô hình (không phải pipeline), cần transform X trước bằng bước prep đã fit.

Kết quả mong đợi: RMSE test thấp nhất khoảng 46.023 ở 497 cây, nên lỗi test vẫn còn giảm rất chậm ở cuối. Với 500 cây, RMSE train khoảng 34.162 và RMSE test khoảng 46.028.

Câu hỏi: lỗi test vẫn giảm ở cây thứ 500. Bạn có nên thử 1000 cây không? Chênh lệch giữa lỗi train và lỗi test nói gì?

In [ ]:
# Viết code ở đây:
# 1. Lấy model = gb.named_steps['model'], Xtr = gb.named_steps['prep'].transform(X_train), Xte tương tự
# 2. Tính RMSE train và test cho mỗi stage bằng staged_predict
# 3. Vẽ hai đường, in RMSE test nhỏ nhất và số cây tương ứng


## Bước 15: Permutation importance

Mục tiêu: xem feature nào quan trọng nhất với Gradient Boosting.

Kiến thức: permutation_importance xáo trộn từng cột của X_test rồi xem lỗi tăng bao nhiêu. Gọi trên cả pipeline gb nên các cột là cột gốc (trước one-hot). Bước này mất khoảng 30 giây.

Kết quả mong đợi: longitude (khoảng 107.000) và latitude (khoảng 107.000) đứng đầu, sau đó là median_income (khoảng 34.560). total_rooms và population gần như không quan trọng (dưới 1.000).

In [ ]:
# Viết code ở đây:
# 1. Gọi permutation_importance(gb, X_test, y_test, n_repeats=5, random_state=42, scoring='neg_root_mean_squared_error')
# 2. Tạo Series importances_mean với index = X_test.columns, sắp xếp giảm dần
# 3. In ra (làm tròn 0 chữ số). Giá trị là số USD mà RMSE tăng lên khi cột bị xáo trộn


## Bước 16: Dự đoán và giá trị thật

Mục tiêu: xem mô hình sai nhiều ở đâu.

Kiến thức: scatter giữa giá thật và giá dự đoán. Nếu đúng hoàn toàn, mọi điểm nằm trên đường chéo y = x.

Kết quả mong đợi: các điểm ở giá thật 500.001 tạo thành một cột dọc, mô hình đoán thấp hơn nhiều. RMSE của 179 khu bị chặn khoảng 96.896, còn các khu khác khoảng 42.297.

In [ ]:
# Viết code ở đây:
# 1. Vẽ scatter y_test và pred_gb (alpha=0.2, s=5), thêm đường chéo màu đỏ
# 2. Tính RMSE riêng cho các khu có y_test >= 500001 và các khu còn lại


## Bước 17: Bảng so sánh các mô hình

Mục tiêu: ghép kết quả các mô hình vào một bảng.

Kiến thức: dùng cùng tập test cho mọi mô hình để so sánh công bằng.

Kết quả mong đợi: Linear, Ridge và Lasso gần như bằng nhau (RMSE khoảng 69.100 đến 69.200, R² 0,635). KNN k = 10 có RMSE khoảng 60.942. Gradient Boosting tốt nhất với RMSE khoảng 46.028 và R² khoảng 0,838.

Câu hỏi: Gradient Boosting tốt nhất nhưng khó giải thích hơn. Khi nào bạn vẫn chọn Ridge?

In [ ]:
# Viết code ở đây:
# 1. Tạo dict models gồm: Linear, Ridge (alpha=1), Lasso (alpha=100), KNN (best_k), Gradient Boosting (dùng lại gb đã fit)
# 2. Với mỗi mô hình, fit nếu chưa fit, tính RMSE, MAE, R² trên tập test
# 3. Tạo DataFrame và in ra


## Bước 18: Tự thử nghiệm

Không có code mẫu ở bước này. Hãy tự chọn ít nhất hai việc sau:

1. Dự đoán np.log(y) thay vì y, rồi đổi lại bằng np.exp trước khi tính RMSE. Linear regression có tốt hơn không?
2. Bỏ các khu có giá 500001 khỏi tập train rồi huấn luyện lại Gradient Boosting. RMSE trên các khu còn lại thay đổi thế nào?
3. Thử HistGradientBoostingRegressor (from sklearn.ensemble import HistGradientBoostingRegressor). Nó nhanh hơn bao nhiêu?
4. Viết ba câu tiếng Anh tóm tắt bảng ở bước 17. Ví dụ: "Gradient boosting reduced the test RMSE from ... (linear regression) to ... USD."
